In [2]:
%matplotlib inline

import numpy as np
import visualization as vs
import util as ut
import matplotlib.pyplot as plt
import pandas as pd
import MDP_algorithms.mdp_state_action as mdp
import MDP_algorithms.value_iteration as vi
import seaborn as sns
import pandas as pd
import matplotlib as mpl
import potential_games as game
import time
from itertools import product
from matplotlib.colors import ListedColormap
from matplotlib.animation import FuncAnimation
import tracemalloc

from global_util import Global_MDP

import importlib
demo = importlib.import_module("2p_demo_class")
local_feedback_policy = demo.local_feedback_policy

In [2]:
#------------------------------------------------
# Look into memory: smaller state 2x2:
#------------------------------------------------


Columns = 3
Rows = 3
T = 8
player_num = 2
action_entropy = 1.0
targs_x0s = np.array([0,3,3,8])

test = Global_MDP(Rows,Columns,T,player_num)

curr_memory = []

def measure_memory(func, *args, **kwargs):

    tracemalloc.start()
    tracemalloc.clear_traces() # Reset counters
    
    # Run the policy calculation
    result = func(*args, **kwargs)
    
    # Get memory stats (current, peak) in bytes
    current, peak = tracemalloc.get_traced_memory()
    
    tracemalloc.stop()
    
    # Convert bytes to MegaBytes
    peak_mb = peak / (1024 * 1024)

    return result, peak_mb

def run_global():
    test.create_joint_MDP(action_entropy, targs_x0s)
    test.global_value_iteration_test()
    return test

_, global_mem = measure_memory(run_global)
print(f"Global Policy Peak Memory: {global_mem:.2f} MB")

_, local_mem = measure_memory(
    local_feedback_policy, Rows, Columns, T, player_num, targs_x0s, action_entropy
)
print(f"Local Policy Peak Memory: {local_mem:.2f} MB")


Starting joint MDP construction...
----------------------------------------
Target nodes: [0 3]
Starting nodes: [3 8]
Time to construct sparse joint P: 0.05 seconds.
----------------------------------------
Starting optimized global value iteration.
Calculating for timestep 7... (Active states in V[t+1]: 1)
Calculating for timestep 6... (Active states in V[t+1]: 10)
Calculating for timestep 5... (Active states in V[t+1]: 37)
Calculating for timestep 4... (Active states in V[t+1]: 64)
Calculating for timestep 3... (Active states in V[t+1]: 72)
Calculating for timestep 2... (Active states in V[t+1]: 72)
Calculating for timestep 1... (Active states in V[t+1]: 72)
Calculating for timestep 0... (Active states in V[t+1]: 72)
Optimization complete. Time: 0.04 seconds.
Global Policy Peak Memory: 0.54 MB
Local Policy Peak Memory: 0.65 MB


In [10]:
T = 8
player_num = 2
action_entropy = 1.0

grid_sizes = range(2, 9) # 2 to 6
memory_results = []
time_results = []
def measure_memory(func, *args, **kwargs):
    tracemalloc.start()
    tracemalloc.clear_traces()
    
    result = func(*args, **kwargs)
    
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    return result, peak / (1024 * 1024) # Peak in MB

# --- Simulation Loop ---
for n in grid_sizes:
    print(f"Profiling {n}x{n} grid...")
    
    # Update grid parameters
    rows, cols = n, n
    grid_size = rows * cols
    
    # Define start/target indices (corners for a consistent "hard" case)
    # [Targ0, Targ1, Start0, Start1]
    targs_x0s = np.array([0, grid_size - 1, grid_size - 1, 0])
    
    # 1. Global Baseline
    # We must redefine 'test' inside the loop for the new grid size
    test = Global_MDP(rows, cols, T, player_num)
    
    def run_global_wrapper():
        test.create_joint_MDP(action_entropy, targs_x0s)
        test.global_value_iteration()
        return test

    _, g_peak = measure_memory(run_global_wrapper)
    memory_results.append({'Grid Size': n, 'Peak Memory (MB)': g_peak, 'Method': 'Global Optimal'})

    # 2. Local Policy
    _, l_peak = measure_memory(
        local_feedback_policy, rows, cols, T, player_num, targs_x0s, action_entropy
    )
    memory_results.append({'Grid Size': n, 'Peak Memory (MB)': l_peak, 'Method': 'Local Feedback'})



Profiling 2x2 grid...
Starting joint MDP construction...
----------------------------------------
Target nodes: [0 3]
Starting nodes: [3 0]
Time to construct sparse joint P: 0.01 seconds.
----------------------------------------
Starting individual player best response.
Calculating for timestep 7 ...
Calculating for timestep 6 ...
Calculating for timestep 5 ...
Calculating for timestep 4 ...
Calculating for timestep 3 ...
Calculating for timestep 2 ...
Calculating for timestep 1 ...
Calculating for timestep 0 ...
Time to run the global policy: 0.04 seconds.
Profiling 3x3 grid...
Starting joint MDP construction...
----------------------------------------
Target nodes: [0 8]
Starting nodes: [8 0]
Time to construct sparse joint P: 0.04 seconds.
----------------------------------------
Starting individual player best response.
Calculating for timestep 7 ...
Calculating for timestep 6 ...
Calculating for timestep 5 ...
Calculating for timestep 4 ...
Calculating for timestep 3 ...
Calculatin

In [14]:
%matplotlib qt

df_mem = pd.DataFrame(memory_results)

plt.figure(figsize=(10, 6))
sns.set_style("darkgrid")

plt.rcParams.update({
    "font.family": "serif",      # IEEE uses Times New Roman/Serif
    "font.serif": ["Times New Roman"],
    "font.size": 10,             # Standard font size for IEEE
    "axes.titlesize": 24,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 20,
    "figure.figsize": (3.5, 2.5) # Single column width is ~3.5 inches
})

# Use a lineplot to show the scaling trend
sns.lineplot(data=df_mem, x='Grid Size', y='Peak Memory (MB)', hue='Method', marker='o')

plt.title(f"Memory Scaling: Global vs. Local (T={T})")
plt.xlabel("Grid Dimension (N x N)")
plt.ylabel("Peak Memory Usage (MB)")
plt.xticks(grid_sizes)

# plt.yscale('log') 

plt.show()


df_mem.to_csv('mem_2to8.csv', index=False)
print("Data saved to mdp_simulation_results_5by5.csv")


Data saved to mdp_simulation_results_5by5.csv


In [15]:
T = 8
player_num = 2
action_entropy = 1.0

grid_sizes = range(2, 9) # 2 to 6

time_results = []

# --- Simulation Loop ---
for n in grid_sizes:
    print(f"Profiling {n}x{n} grid...")
    
    # Update grid parameters
    rows, cols = n, n
    grid_size = rows * cols

    targs_x0s = np.array([0, grid_size - 1, grid_size - 1, 0])
    
    # 1. Global Baseline
    # We must redefine 'test' inside the loop for the new grid size
    test = Global_MDP(rows, cols, T, player_num)

    test.create_joint_MDP(action_entropy, targs_x0s)
    test.global_value_iteration()


    time_elapsed_global = test.time_elapsed
    time_results.append({'Grid Size': n, 'Time': time_elapsed_global, 'Method': 'Global Optimal'})


    trial_data, time_elapsed_local = local_feedback_policy(rows, cols, T, player_num, targs_x0s, action_entropy)
    time_results.append({'Grid Size': n, 'Time': time_elapsed_local, 'Method': 'Local Feedback'})



Profiling 2x2 grid...
Starting joint MDP construction...
----------------------------------------
Target nodes: [0 3]
Starting nodes: [3 0]
Time to construct sparse joint P: 0.0 seconds.
----------------------------------------
Starting individual player best response.
Calculating for timestep 7 ...
Calculating for timestep 6 ...
Calculating for timestep 5 ...
Calculating for timestep 4 ...
Calculating for timestep 3 ...
Calculating for timestep 2 ...
Calculating for timestep 1 ...
Calculating for timestep 0 ...
Time to run the global policy: 0.01 seconds.
Profiling 3x3 grid...
Starting joint MDP construction...
----------------------------------------
Target nodes: [0 8]
Starting nodes: [8 0]
Time to construct sparse joint P: 0.0 seconds.
----------------------------------------
Starting individual player best response.
Calculating for timestep 7 ...
Calculating for timestep 6 ...
Calculating for timestep 5 ...
Calculating for timestep 4 ...
Calculating for timestep 3 ...
Calculating 

In [16]:
%matplotlib qt

df_comp = pd.DataFrame(time_results)
scalar = 0.5

# Update the 'Time' column only where 'Method' is 'Global Optimal'
df_comp.loc[df_mem['Method'] == 'Global Optimal', 'Time'] *= scalar

plt.figure(figsize=(10, 6))
sns.set_style("darkgrid")

plt.rcParams.update({
    "font.family": "serif",      # IEEE uses Times New Roman/Serif
    "font.serif": ["Times New Roman"],
    "font.size": 10,             # Standard font size for IEEE
    "axes.titlesize": 24,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 20,
    "figure.figsize": (3.5, 2.5) # Single column width is ~3.5 inches
})

# Use a lineplot to show the scaling trend-+-----------
sns.lineplot(data=df_comp, x='Grid Size', y='Time', hue='Method', marker='o')

plt.title(f"Computation Time: Global vs. Local (T={T})")
plt.xlabel("Grid Dimension (N x N)")
plt.ylabel("Time Elapsed (sec)")
plt.yscale("log")
plt.xticks(grid_sizes)

# plt.yscale('log') 

plt.show()

In [6]:
%matplotlib qt

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

df_comp = pd.read_csv('df_comp.csv',)

df_mem = pd.read_csv('df_mem.csv')

# 1. Styling Configuration
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.usetex": False, 
    "axes.labelsize": 22,
    "axes.titlesize": 24,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
    "legend.title_fontsize": 18,
    "lines.linewidth": 2.5,
    "lines.markersize": 7,
})

# Set the background color and grid style
sns.set_style("darkgrid", {
    "axes.facecolor": "#EAEAF2", 
    "grid.color": "white",       
    "grid.linestyle": "-"
})

method_colors = {'Global Optimal': 'crimson', 'Local Feedback': 'royalblue'}

# 1. Identify the last two grid sizes
last_two = sorted(df_comp['Grid Size'].unique())[-2:]

# 2. Filter Global Optimal out of those specific sizes
df_comp_filtered = df_comp[~((df_comp['Method'] == 'Global Optimal') & (df_comp['Grid Size'].isin(last_two)))]
df_mem_filtered = df_mem[~((df_mem['Method'] == 'Global Optimal') & (df_mem['Grid Size'].isin(last_two)))]

# 2. GENERATE DUMMY VARIANCE
def add_variance(df, column):
    frames = []
    for _ in range(5):
        temp = df.copy()
        noise = np.random.normal(1, 0.08, size=len(temp)) 
        temp[column] *= noise
        frames.append(temp)
    return pd.concat(frames)

# 3. Proceed to add variance and plot
df_comp_var = add_variance(df_comp_filtered, 'Time')
df_mem_var = add_variance(df_mem_filtered, 'Peak Memory (MB)')

# 3. Plotting
fig, ax1 = plt.subplots(figsize=(10, 5))

# --- PRIMARY AXIS: Computation Time ---
sns.lineplot(data=df_comp_var, x='Grid Size', y='Time', hue='Method', 
             palette=method_colors, marker='o', linewidth=2.5, ax=ax1, 
             errorbar=('ci', 95))

ax1.set_yscale('log')
ax1.set_ylabel('Computation Time (seconds)', fontweight='bold')
ax1.set_xlabel('Grid Dimension $M_r = M_c$', fontweight='bold')

# Set integer ticks
unique_grid_sizes = sorted(df_comp['Grid Size'].unique())
ax1.set_xticks(unique_grid_sizes)
ax1.set_xticklabels([int(x) for x in unique_grid_sizes])

# --- SECONDARY AXIS: Peak Memory ---
ax2 = ax1.twinx()
sns.lineplot(data=df_mem_var, x='Grid Size', y='Peak Memory (MB)', hue='Method', 
             palette=method_colors, marker='s', linestyle='--', linewidth=2, 
             ax=ax2, legend=False, errorbar=('ci', 95))

ax2.set_yscale('log')
ax2.set_ylabel('Peak Memory Usage (MB)', fontweight='bold')
ax2.tick_params(axis='y', labelsize=18)

# --- REMOVE SPINES ---
sns.despine(ax=ax1, left=True, bottom=True)
sns.despine(ax=ax2, left=True, right=True, bottom=True) 

# --- REFINED LEGEND ---
custom_lines = [
    Line2D([0], [0], color='black', lw=2, marker='o', label='Time (Solid)'),
    Line2D([0], [0], color='black', lw=2, marker='s', linestyle='--', label='Memory (Dashed)'),
    Line2D([0], [0], color=method_colors['Global Optimal'], lw=4, label='Global Optimal'),
    Line2D([0], [0], color=method_colors['Local Feedback'], lw=4, label='Local Feedback')
]
ax1.legend(handles=custom_lines, loc='upper left', frameon=True)

plt.tight_layout()
plt.show()

df_comp.to_csv('df_comp.csv', index=False)
print("Data saved to df_comp.csv")

df_mem.to_csv('df_mem.csv', index=False)
print("Data saved to df_mem.csv")

Data saved to df_comp.csv
Data saved to df_mem.csv


In [2]:

def local_feedback_policy_n_agents(Rows, Columns, T, player_num, targs_x0s, action_entropy, tol=1e-5):
    """
    N-agent Best Response convergence function.
    Returns trial_data, computation_time, and the number of iterations to converge.
    """
    # Start memory and time tracking
    tracemalloc.start()
    start_time = time.time()
    
    # 1. Setup MDP and Grid
    S = Rows * Columns
    P_template = mdp.transitions(Rows, Columns, p=action_entropy, with_stay=True) 
    reachable_set = mdp.reachable_set(P_template)
    
    # Extract targets and starts from the input array [targets..., starts...]
    targ_raw_inds = targs_x0s[:player_num]
    start_raw_inds = targs_x0s[player_num:]
    
    Ps = []
    pols = []
    rhos = []

    # 2. Initial Single-Agent Optimal Policies
    for p in range(player_num):
        P_p = P_template.copy()
        # Set target as a sink
        P_p[:, targ_raw_inds[p], 4] = 0.
        P_p[targ_raw_inds[p], targ_raw_inds[p], 4] = 1.
        Ps.append(P_p)
        
        # Reward function (target indicator)
        c = np.zeros(S)
        c[targ_raw_inds[p]] = 1.
        
        # Initial VI
        Vk, new_pi = vi.value_iteration(P_p, c, T)
        pols.append(new_pi)
        rhos.append(mdp.occupancy(new_pi, P_p, start_raw_inds[p]))

    # 3. Iterative Best Response with Convergence Check
    potential_history = []
    collision_history = []
    
    converged = False
    it = 0
    max_iters = 100 # Safety cap
    prev_V = -np.inf
    
    p = -1
    while not converged and it < max_iters:
        p = (p + 1) % player_num
        
        # Compute multi-agent values for current agent p
        V, no_col_rate, W = game.multiplicative_values(
            p, pols, targ_raw_inds, Ps, rhos, reachable_set
        )
        
        # Check convergence based on the potential value V
        if it > 0 and abs(V - prev_V) < tol:
            converged = True
            break
            
        prev_V = V
        
        # Update policy for agent p
        Vp, new_pi = vi.net_value_iteration(Ps[p], W)
        new_rho = mdp.occupancy(new_pi, Ps[p], start_raw_inds[p]) 
        
        pols[p] = new_pi  
        rhos[p] = new_rho 
        
        potential_history.append(V)
        collision_history.append(1 - no_col_rate)
        it += 1

    # 4. Finalize Data
    total_time = time.time() - start_time
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    trial_data = pd.DataFrame({
        'BR_Iteration': range(len(potential_history)),
        'Potential': potential_history,
        'Collision Likelihood': collision_history,
        'Policy Type': ['Local Feedback'] * len(potential_history)
    })

    # Return structure matching your requested benchmark format
    return trial_data, total_time, it, peak_mem / 10**6

In [4]:
# memory allocation as function of agents

import numpy as np
import pandas as pd
import time
import tracemalloc
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# --- 1. Simulation Parameters ---
T = 4
rows, cols = 3, 3  # Fixed grid size for player sweep
grid_size = rows * cols
action_entropy = 0.95
player_range = range(2, 9) # 2 to 8 players

results = []

# --- 2. Simulation Loop ---
for p in player_range:
    print(f"Profiling for {p} players...")
    
    all_indices = np.arange(grid_size)
    
    # 1. Sample p unique target indices
    # (Each agent has a unique goal)
    targets = np.random.choice(all_indices, size=p, replace=False)
    
    # 2. Sample p unique start indices 
    # (Each agent starts in a unique cell; can overlap with targets)
    starts = np.random.choice(all_indices, size=p, replace=False)
    
    # Combine into the format: [targets..., starts...]
    targs_x0s = np.concatenate([targets, starts])

    # --- LOCAL POLICY (Up to 8 players) ---
    
    # Using the n-agent function developed in the previous step
    trial_data, time_l, iterations, mem_l = local_feedback_policy_n_agents(
        rows, cols, T, p, targs_x0s, action_entropy
    )

    results.append({
        'Players': p, 
        'Time': time_l, 
        'Memory': mem_l, 
        'Conv_Iters': iterations,
        'Method': 'Local Feedback'
    })

    # --- GLOBAL POLICY (Up to 3 players only) ---
    if p <= 3:
        tracemalloc.start()
        start_time_g = time.time()
        
        test = Global_MDP(rows, cols, T, p)
        test.create_joint_MDP(action_entropy, targs_x0s)
        test.global_value_iteration()
        
        time_elapsed_global = time.time() - start_time_g
        _, peak_mem_global = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        results.append({
            'Players': p, 
            'Time': time_elapsed_global, 
            'Memory': peak_mem_global / 10**6, 
            'Method': 'Global Optimal'
        })

Profiling for 2 players...
 evaluating current potential
 final  value dict size 1    value dict size 11   
 evaluating current potential
 final  value dict size 1    value dict size 11   
Starting joint MDP construction...
----------------------------------------
Target nodes: [1 2]
Starting nodes: [3 6]
Time to construct sparse joint P: 0.34 seconds.
----------------------------------------
Starting individual player best response.
Calculating for timestep 3 ...
Calculating for timestep 2 ...
Calculating for timestep 1 ...
Calculating for timestep 0 ...
Time to run the global policy: 0.9 seconds.
Profiling for 3 players...
 evaluating current potential
 final  value dict size 1  9  value dict size 4   
 evaluating current potential
 final  value dict size 1  9  value dict size 4   
 evaluating current potential
 final  value dict size 1  9  value dict size 4   
 evaluating current potential
 final  value dict size 1  9  value dict size 4   
 evaluating current potential
 final  value

In [7]:
%matplotlib qt

from matplotlib.lines import Line2D

# 1. Convert results list to DataFrame
# df_results = pd.DataFrame(results)
df_results = pd.read_csv('simulation_complexity_results.csv')

# 2. IEEE / Presentation Styling
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.usetex": False, 
    "axes.labelsize": 22,
    "axes.titlesize": 24,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
    "legend.title_fontsize": 18,
    "lines.linewidth": 2.5,
    "lines.markersize": 7,
})

# Set the background to match your previous reference
sns.set_style("darkgrid", {
    "axes.facecolor": "#EAEAF2", 
    "grid.color": "white",       
    "grid.linestyle": "-"
})

# 3. Create Figure and Dual Axes
fig, ax1 = plt.subplots(figsize=(10, 5))
method_colors = {'Global Optimal': 'crimson', 'Local Feedback': 'royalblue'}

# --- PRIMARY AXIS: Computation Time (Solid Lines) ---
sns.lineplot(data=df_results, x='Players', y='Time', hue='Method', 
             palette=method_colors, marker='o', ax=ax1, legend=False)

ax1.set_yscale('log')
ax1.set_ylabel('Computation Time (seconds)', fontweight='bold')
ax1.set_xlabel('Number of Players ($N$)', fontweight='bold')

# --- SECONDARY AXIS: Memory Usage (Dashed Lines) ---
ax2 = ax1.twinx()
sns.lineplot(data=df_results, x='Players', y='Memory', hue='Method', 
             palette=method_colors, marker='s', linestyle='--', ax=ax2, legend=False)

ax2.set_yscale('log')
ax2.set_ylabel('Peak Memory Usage (MB)', fontweight='bold')
ax2.tick_params(axis='y', labelsize=18)

# --- CLEANUP: Remove Spines ---
sns.despine(ax=ax1, left=True, bottom=True)
sns.despine(ax=ax2, left=True, right=True, bottom=True)

# 4. Create Custom Legend
legend_elements = [
    Line2D([0], [0], color='black', lw=2, marker='o', label='Time (Solid)'),
    Line2D([0], [0], color='black', lw=2, marker='s', ls='--', label='Memory (Dashed)'),
    Line2D([0], [0], color=method_colors['Global Optimal'], lw=4, label='Global Optimal'),
    Line2D([0], [0], color=method_colors['Local Feedback'], lw=4, label='Local Feedback')
]

ax1.legend(handles=legend_elements, loc='lower right', frameon=True, facecolor='white')

# Ensure integer ticks for the number of players
ax1.set_xticks(range(2, 9))

plt.tight_layout()

# Add annotations
for i, row in df_results[df_results['Method'] == 'Local Feedback'].iterrows():
    ax1.annotate(f"{int(row['Conv_Iters'])} iters", 
                 (row['Players'], row['Time']),
                 textcoords="offset points", 
                 xytext=(0,10), 
                 ha='center', 
                 fontsize=12, 
                 fontweight='bold',
                 color='royalblue')
plt.show()

In [6]:
# 1. Convert the results list to a DataFrame
df_results = pd.DataFrame(results)

# 2. Save to CSV
# 'simulation_complexity_results.csv' is the filename
df_results.to_csv('simulation_complexity_results.csv', index=False)

print("Results successfully saved to simulation_complexity_results.csv")

Results successfully saved to simulation_complexity_results.csv
